# w21b_gpu2_multidataset_jrdb_jta_jaad_eth_dut_imptc
Inputs: `shortpose-wave1-private-inputs` + `w1-gpu2-dut-pose-and-graph` | Accelerator: `GPU T4 x2`


In [ ]:
import hashlib, os, subprocess, sys, tempfile, zipfile
from pathlib import Path
import torch

WORK, MOUNT = Path('/kaggle/working'), Path('/kaggle/input')
assert WORK.is_dir() and torch.cuda.is_available(), 'Kaggle GPU required'
n_gpu = torch.cuda.device_count()
REV = '3c276979be4b4d04e657c23f6b53a1f5ef00768a'
with tempfile.TemporaryDirectory(prefix='w21_g2_') as ts:
    tmp = Path(ts)
    repo, sc0, sc1 = tmp / 'repo', tmp / 'sc0', tmp / 'sc1'
    subprocess.run(['git', 'init', '-q', str(repo)], check=True)
    subprocess.run(['git', '-C', str(repo), 'fetch', '-q', '--depth=1', 'https://github.com/pjyyyds/Unstructured-ShortPose-Pred.git', REV], check=True)
    subprocess.run(['git', '-C', str(repo), 'checkout', '-q', '--detach', 'FETCH_HEAD'], check=True)
    base_env = {**os.environ, 'PYTHONPATH': str(repo), 'PYTHONDONTWRITEBYTECODE': '1'}
    dut_args = []
    z_hits = list(MOUNT.rglob('dut_shared_space_graph_transfer.zip'))
    pg_hits = list(MOUNT.rglob('private_graph.npz'))
    et_hits = list(MOUNT.rglob('errors_and_targets.npz'))
    if z_hits:
        dut_args = ['--dut-zip', str(z_hits[0])]
    elif pg_hits and et_hits:
        dut_zip = tmp / 'dut_graph.zip'
        with zipfile.ZipFile(dut_zip, 'w', zipfile.ZIP_DEFLATED) as dz:
            dz.write(pg_hits[0], 'private_graph.npz')
            dz.write(et_hits[0], 'errors_and_targets.npz')
        dut_args = ['--dut-zip', str(dut_zip)]
    imptc_tar = tmp / 'imptc_samples.tar.gz'
    d0 = WORK / 'wave21_jrdb_and_jta_emergence_maneuvers.zip'
    d1 = WORK / 'wave21_jaad_deocclusion_and_eth_ucy_6dataset.zip'
    e0 = {**base_env, 'CUDA_VISIBLE_DEVICES': '0'}
    e1 = {**base_env, 'CUDA_VISIBLE_DEVICES': '1' if n_gpu > 1 else '0'}
    p0 = subprocess.Popen([
        sys.executable, '-u', '-m', 'scripts.run_wave21_jrdb_and_jta_emergence_maneuvers',
        '--repo-root', str(repo), '--scratch-dir', str(sc0), '--zip-path', str(d0)
    ], cwd=repo, env=e0)
    p1 = subprocess.Popen([
        sys.executable, '-u', '-m', 'scripts.run_wave21_jaad_deocclusion_and_eth_ucy_6dataset',
        '--scratch-dir', str(sc1), '--imptc-tar', str(imptc_tar), '--zip-path', str(d1), *dut_args
    ], cwd=repo, env=e1)
    rc0, rc1 = p0.wait(), p1.wait()
    assert rc0 == 0 and rc1 == 0, f'Subprocess failed: rc0={rc0}, rc1={rc1}'
    for d in (d0, d1):
        print('DONE:', d, d.stat().st_size, hashlib.sha256(d.read_bytes()).hexdigest(), flush=True)
